# teach-aws: Malay AWS QA — Using the Final Model

This notebook shows how to **use** the finetuned Qwen3.5-2B model (LoRA merged):
- what it does well (accurate AWS answers in Bahasa Malaysia)
- what it gets wrong (occasional wrong-feature recall — hallucination)
- the grounding guardrail we ship to handle that

Training code is private; this is inference-only.
Model weights: `hf:ORG/teach-aws-qwen3.5-2b` (placeholder — replace with the published repo id).

In [ ]:
# 1) Install (Colab GPU runtime)
%pip install -q vllm huggingface_hub

In [ ]:
# 2) Load the model (vLLM for fast batched inference)
from vllm import LLM, SamplingParams

MODEL = 'hf:ORG/teach-aws-qwen3.5-2b'  # TODO: replace with published repo id
llm = LLM(model=MODEL, max_model_len=4096, gpu_memory_utilization=0.85, max_num_seqs=256, enforce_eager=True)

In [ ]:
# 3) Ask questions
questions = [
    'Bolehkah S3 Transfer Acceleration melengkapi AWS Direct Connect?',
    'Bagaimanakah Step Functions menyokong keselarian?',
    'Apakah perbezaan antara Amazon S3 dan Amazon EFS?',
]
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template([{'role':'user','content':q}], tokenize=False, add_generation_prompt=True) for q in questions]
outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=2048))
for q, o in zip(questions, outs):
    print('Q:', q)
    print('A:', o.outputs[0].text.strip()[:400])
    print('---')

## The failure mode: confident hallucination

Example from our eval (real generation, judged FALSE):

> **Q:** Bolehkah saya menyimpan data saya apabila stack dipadamkan?
>
> **A:** "Ya, anda boleh menggunakan ciri **Memory and Cache Behavior** dalam AWS CloudFormation..." ← feature does not exist; correct answer is a deletion policy with snapshots.

76% of the model's failures are fluent, authoritative-sounding text like this. Users cannot self-detect them.

In [ ]:
# 4) The grounding guardrail: flag answers mentioning entities absent from the reference
import re

ENT_RE = re.compile(
    r"\b(?:Amazon\s+[A-Z][a-zA-Z0-9]+|AWS\s+[A-Z][a-zA-Z0-9]+(?:\s+[A-Z][a-zA-Z0-9]+){0,2}"
    r"|[A-Z][a-zA-Z0-9]*(?:Flow|Front|Watch|Maker|Bridge|Formation|Sync|Lake|Base|Glue|Grid)"
    r"|[A-Z][a-zA-Z0-9]+\s(?:and\s)?[A-Z][a-zA-Z0-9]+(?:\s[A-Z][a-zA-Z0-9]+)*"
    r"|S3|EC2|EBS|Lambda|IAM|VPC|KMS|SSE|TLS|SSL|HTTP|DNS|SQL|API|SDK|CLI|MFA|HSM|WAF"
    r"|Route\s?53|Step\sFunctions|CloudFormation|CloudFront|CloudWatch|Direct\sConnect)\b"
)

def entities(text):
    return {m.group(0).lower().replace('amazon ','').replace('aws ','') for m in ENT_RE.finditer(text)}

def guard(answer, reference):
    novel = entities(answer) - entities(reference)
    if not novel:
        return answer
    return answer + '\n\n---\n⚠️ Unverified entities mentioned: ' + ', '.join(sorted(novel)) + '. Please verify with AWS docs.'

# demo
ref = 'CloudFormation membolehkan anda menentukan deletion policy untuk sumber dalam templat. Snapshot dicipta untuk volum Amazon EBS.'
hallucinated = 'Ya, anda boleh menggunakan ciri Memory and Cache Behavior dalam AWS CloudFormation untuk mengekalkan data.'
print(guard(hallucinated, ref))

## Guardrail benchmark (from our eval, n=343)

- Catches **59%** of wrong answers (22/37) — they mention entities not in the reference
- False-flags only **0.3%** of correct answers (1/306)
- Inference-only: no model change, works with any generation

Deployment pattern: retrieve canonical answer (your KB/doc) → generate freely → verify entities → warn or fallback when flagged.